# Building with Small Language Models (SLMs)

A practical guide to working with IBM Granite 4 and other Small Language Models.

For detailed explanations, see: [`building_slm.md`](../.../../building_slm.md)

## What You'll Learn

1. Setup and connect to Granite 4 via Ollama
2. Few-shot prompting techniques
3. Structured prompt engineering
4. RAG for SLMs
5. Tool calling patterns
6. Subagent orchestration
7. Programmatic hybrid patterns
8. Migration strategies (LLM → SLM)
9. Document classification example
10. Fine-tuning with LoRA

## 1. Environment Setup

Install required libraries:

In [ ]:
! echo "::group::Install Dependencies"
%pip install -q langchain-openai langchain langgraph \
    sentence-transformers faiss-cpu \
    datasets pandas
! echo "::endgroup::"

## 2. Connect to Granite 4

**Setup:** Uses Ollama locally with `granite3.1-dense:2b`

**Install:** `ollama pull granite3.1-dense:2b`

**Fallback:** Automatic mock mode if Ollama unavailable

In [ ]:
import os
import sys
from langchain_openai import ChatOpenAI

# ── Environment Check ──────────────────────────────────────────────────────
print("🔍 Environment Check")
print(f"Python: {sys.version.split()[0]}")
print(f"Working Directory: {os.getcwd()}")

# ── API Connection with Fallback ───────────────────────────────────────────
def setup_llm():
    """Setup LLM with automatic fallback: Ollama → Mock."""
    
    # Try Ollama local
    try:
        import requests
        resp = requests.get("http://localhost:11434/api/tags", timeout=2)
        if resp.status_code == 200:
            print("  Run: ollama pull granite4:350m (if not already installed)")
            return ChatOpenAI(
                model_name="granite4:350m",
                api_key="ollama",
                base_url="http://localhost:11434/v1",
                temperature=0.0,
                max_tokens=512,
            )
    except Exception as e:
        print(f"\n⚠️  Ollama not available: {e}")
    
llm = setup_llm()

# Quick smoke test
try:
    response = llm.invoke("In one sentence, what is a Small Language Model?")
    print("\n✓ Connection successful!")
    print(f"Response: {response.content[:150]}...")
except Exception as e:
    print(f"\n⚠️  Error: {e}")
    print("Continuing with mock responses for demonstration...")

## 3. Few-Shot Prompting

**Key technique:** Provide 3-10 input/output examples for pattern matching.

 Details: See [In-Context Learning section](../../building_slm.md#in-context-learning--few-shot-prompting) in building_slm.md

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
import json

# Static few-shot: named entity extraction
# The format is consistent: input text → structured JSON output
# Notice the edge case — "no attendees" — teaching the model to return empty lists
FEW_SHOT_EXAMPLES = """
Text: "John Smith visited Paris last summer."
Output: {"persons": ["John Smith"], "locations": ["Paris"]}

Text: "The conference in Tokyo was attended by Dr. Sarah Chen and Prof. Michael Brown."
Output: {"persons": ["Dr. Sarah Chen", "Prof. Michael Brown"], "locations": ["Tokyo"]}

Text: "Alice went to London and met Bob at the museum."
Output: {"persons": ["Alice", "Bob"], "locations": ["London"]}

Text: "The meeting had no attendees."
Output: {"persons": [], "locations": []}
"""

def extract_entities(text: str) -> str:
    """Extract entities with error handling."""
    prompt = f"""Extract person names and locations from the text. Return ONLY valid JSON.

{FEW_SHOT_EXAMPLES}
Text: "{text}"
Output:"""
    try:
        response = llm.invoke(prompt).content
        # Validate JSON
        json.loads(response)
        return response
    except json.JSONDecodeError:
        # Extract JSON from response if wrapped in text
        import re
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            return match.group()
        return '{"persons": [], "locations": []}'
    except Exception as e:
        print(f"⚠️  Error: {e}")
        return '{"persons": [], "locations": []}'


test_inputs = [
    "Mary Johnson and her colleague traveled to Berlin for the summit.",
    "The CEO of Acme Corp, David Lee, signed the deal in Singapore.",
    "No one attended the event in New York.",
]

print("=== Static Few-Shot Entity Extraction ===")
for text in test_inputs:
    result = extract_entities(text)
    print(f"Input : {text}")
    print(f"Output: {result}\n")

In [ ]:
# Dynamic few-shot: select the most relevant examples using embedding similarity
# This outperforms static examples for diverse input distributions
from sentence_transformers import SentenceTransformer
import numpy as np

# Example pool — in production, load from a curated database
EXAMPLE_POOL = [
    {"input": "John Smith visited Paris.",              "output": '{"persons":["John Smith"],"locations":["Paris"]}'},
    {"input": "Dr. Chen attended the Tokyo summit.",    "output": '{"persons":["Dr. Chen"],"locations":["Tokyo"]}'},
    {"input": "Alice and Bob met in London.",           "output": '{"persons":["Alice","Bob"],"locations":["London"]}'},
    {"input": "The report was filed with no names.",    "output": '{"persons":[],"locations":[]}'},
    {"input": "CEO Maria Gonzalez flew to Berlin.",     "output": '{"persons":["Maria Gonzalez"],"locations":["Berlin"]}'},
    {"input": "Prof. Brown spoke at the Rome conference.", "output": '{"persons":["Prof. Brown"],"locations":["Rome"]}'},
]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
pool_embeddings = embedder.encode([e["input"] for e in EXAMPLE_POOL])

def select_relevant_examples(query: str, k: int = 3) -> list[dict]:
    """Return the k most semantically similar examples to the query."""
    query_emb = embedder.encode([query])
    scores = np.dot(pool_embeddings, query_emb.T).flatten()
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [EXAMPLE_POOL[i] for i in top_k_idx]

def extract_entities_dynamic(text: str, k: int = 3) -> str:
    examples = select_relevant_examples(text, k=k)
    shots = "\n".join(
        f'Text: "{e["input"]}"\nOutput: {e["output"]}'
        for e in examples
    )
    prompt = f"""Extract person names and locations. Return ONLY valid JSON.\n\n{shots}\n\nText: "{text}"\nOutput:"""
    return llm.invoke(prompt).content

query = "Ambassador Williams landed in Seoul for bilateral talks."
selected = select_relevant_examples(query)
print("Selected examples:", [e["input"] for e in selected])
print("Result:", extract_entities_dynamic(query))

## 4. Retrieval-Augmented Generation (RAG)

**Purpose:** Compensate for limited parametric knowledge with just-in-time context.

Details: See [RAG section](../../building_slm.md#retrieval-augmented-generation-rag) in building_slm.md

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ── Sample knowledge base (replace with your domain documents) ───────────
DOCUMENTS = [
    "Granite 4.0 models use a hybrid Mamba-2 + Transformer architecture that reduces memory by over 70%.",
    "Granite 4.0 H-Small has 32B total parameters with 9B active (MoE), suited for enterprise workflows.",
    "Granite 4.0 H-Micro is a 3B dense model designed for edge and high-throughput scenarios.",
    "All Granite 4.0 models are released under the Apache 2.0 open-source license.",
    "Granite 4.0 supports structured JSON output, tool calling, and RAG-optimized inference.",
    "You can run Granite locally using Ollama: `ollama pull granite3-dense:8b`.",
    "Fine-tuning Granite 4.0 with LoRA on a T4 GPU takes hours, not days.",
    "Granite 4.0 supports 12+ languages and context windows up to 128K tokens.",
]

# Build FAISS vector index
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(DOCUMENTS, convert_to_numpy=True)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])  # inner product = cosine (normalized)
faiss.normalize_L2(doc_embeddings)
index.add(doc_embeddings)

print(f"Index built with {index.ntotal} documents.")


def retrieve(query: str, k: int = 3) -> list[str]:
    """Return the k most relevant document chunks for the query."""
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    _, indices = index.search(q_emb, k)
    return [DOCUMENTS[i] for i in indices[0]]

In [ ]:
# ── RAG-powered QA ────────────────────────────────────────────────────────
RAG_SYSTEM = (
    "You are a helpful assistant. "
    "Answer the question using ONLY the provided context. "
    "If the answer is not in the context, say: 'I don't have enough information.'"
)

# Few-shot examples teaching citation behavior
RAG_FEW_SHOT = """
Context: "The Granite 4.0 models use a hybrid Mamba-2 architecture."
Question: What architecture does Granite 4.0 use?
Answer: Granite 4.0 models use a hybrid Mamba-2 architecture.

Context: "IBM released Granite models under Apache 2.0 license."
Question: What is the pricing for Granite?
Answer: I don't have enough information about pricing, but the models are Apache 2.0 open source.
"""

def rag_answer(question: str, k: int = 3) -> dict:
    context_docs = retrieve(question, k=k)
    context_str = "\n".join(f"- {d}" for d in context_docs)

    prompt = f"""{RAG_FEW_SHOT}
Context:
{context_str}

Question: {question}
Answer:"""

    answer = llm.invoke([SystemMessage(RAG_SYSTEM), HumanMessage(prompt)]).content
    return {"question": question, "retrieved": context_docs, "answer": answer}


questions = [
    "How much memory does Granite 4.0 save compared to pure transformers?",
    "Can I fine-tune Granite on a laptop?",
    "What is the maximum context window for Granite 4.0?",
    "What is Granite's training cost in USD?",   # not in documents → should say unknown
]

for q in questions:
    result = rag_answer(q)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"   (retrieved: {len(result['retrieved'])} docs)\n")

## 5. Tool Calling

**Strategy:** Keep tool sets minimal (5-15 tools max), use strict JSON schemas.

Details: See [Tool Calling section](../../building_slm.md#tool-calling--structured-function-invocations) in building_slm.md

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
import json


# ── Tool Definitions ──────────────────────────────────────────────────────
@tool
def search_products(query: str, category: str = "all") -> dict:
    """Search the product catalog. Example: search_products('laptop', 'electronics')

    Args:
        query: Search keywords (e.g. 'laptop', 'running shoes').
        category: Filter by category. One of: 'electronics', 'clothing', 'books', 'all'.
    """
    # Simulated response — replace with real catalog API
    mock_results = {
        "electronics": [{"id": "E1", "name": "ProBook Laptop 15", "price": 899.99}],
        "clothing": [{"id": "C1", "name": "Trail Runner Shoes", "price": 129.99}],
        "books": [{"id": "B1", "name": "AI Engineering Handbook", "price": 49.99}],
    }
    results = mock_results.get(category, [])
    print(f"  [tool] search_products(query={query!r}, category={category!r}) → {len(results)} results")
    return {"results": results, "total": len(results)}


@tool
def get_order_status(order_id: str) -> dict:
    """Retrieve the current status of an order by its ID. Example: get_order_status('ORD-12345')

    Args:
        order_id: The order identifier (format: ORD-XXXXX).
    """
    # Simulated response
    mock_orders = {
        "ORD-12345": {"status": "shipped", "eta": "2026-03-01", "carrier": "FedEx"},
        "ORD-99999": {"status": "processing", "eta": "2026-03-05", "carrier": None},
    }
    result = mock_orders.get(order_id, {"status": "not_found", "eta": None})
    print(f"  [tool] get_order_status(order_id={order_id!r}) → {result}")
    return result


tools = [search_products, get_order_status]
llm_with_tools = llm.bind_tools(tools)


# ── LangGraph Agent ───────────────────────────────────────────────────────
class AgentState(TypedDict, total=False):
    messages: Annotated[list, add_messages]

def call_llm(state: AgentState) -> AgentState:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def route(state: AgentState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END

tool_node = ToolNode(tools=tools)
graph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("tools", tool_node)
graph.add_edge(START, "llm")
graph.add_conditional_edges("llm", route, {"tools": "tools", END: END})
graph.add_edge("tools", "llm")
agent = graph.compile()


def run_agent(user_input: str):
    print(f"User: {user_input}")
    state = {"messages": [HumanMessage(user_input)]}
    for event in agent.stream(state):
        for value in event.values():
            last = value["messages"][-1]
            if isinstance(last, AIMessage) and last.content:
                print(f"Agent: {last.content}")


run_agent("Search for laptops in the electronics category.")
print()
run_agent("What's the status of order ORD-12345?")

## 6. Document Classification Example

**Comparison:** LLM vs SLM approach for ticket classification.

Details: See [Document Classification section](../../building_slm.md#practical-example-document-classification-system) in building_slm.md

In [ ]:
from collections import Counter


class TicketClassificationSystem:
    """Production-ready SLM-based ticket classifier using all six techniques."""

    CATEGORIES = ["Technical Issue", "Billing Question", "Feature Request",
                  "General Inquiry", "Complaint"]

    # Technique 1: Few-shot examples
    FEW_SHOT = """
Ticket: "Login fails with error 500 every morning"
Category: Technical Issue

Ticket: "Double charge appeared on my card yesterday"
Category: Billing Question

Ticket: "Please add CSV export to the dashboard"
Category: Feature Request

Ticket: "Do you offer annual pricing?"
Category: General Inquiry

Ticket: "This is the third time my issue hasn't been resolved"
Category: Complaint
"""

    def __init__(self, llm_client, fallback_llm_client=None):
        self.llm = llm_client
        self.fallback_llm = fallback_llm_client  # Larger model for uncertain cases
        self.metrics = Counter()

    # Technique 6: Code handles pre-processing
    def _preprocess(self, ticket: str) -> str | None:
        ticket = ticket.strip()
        if len(ticket) < 10:
            self.metrics["too_short"] += 1
            return None
        return ticket[:1000]  # Respect SLM context limits

    # Technique 2: Structured prompt engineering
    def _build_prompt(self, ticket: str) -> str:
        categories_str = ", ".join(f'"{c}"' for c in self.CATEGORIES)
        return (
            f"You are a customer support ticket classifier. "
            f"Choose exactly ONE category from: [{categories_str}]. "
            f"Return only the category name.\n\n"
            f"{self.FEW_SHOT}\n"
            f'Ticket: "{ticket}"\n'
            f"Category:"
        )

    # Technique 6: Code handles post-processing + validation
    def _parse_category(self, raw: str) -> str | None:
        raw = raw.strip()
        for cat in self.CATEGORIES:
            if cat.lower() in raw.lower():
                return cat
        return None  # Invalid response → triggers fallback

    def classify(self, ticket: str) -> dict:
        # Step 1: Pre-process (code)
        clean = self._preprocess(ticket)
        if clean is None:
            return {"ticket": ticket, "category": None, "source": "rejected", "error": "too_short"}

        # Step 2: SLM classification
        t0 = time.time()
        raw = self.llm.invoke(self._build_prompt(clean)).content
        latency_ms = (time.time() - t0) * 1000

        # Step 3: Validate (code)
        category = self._parse_category(raw)
        source = "slm"

        # Step 4: Fallback if SLM output is invalid
        if category is None and self.fallback_llm:
            self.metrics["fallback_invoked"] += 1
            raw = self.fallback_llm.invoke(self._build_prompt(clean)).content
            category = self._parse_category(raw) or "General Inquiry"
            source = "fallback_llm"

        category = category or "General Inquiry"  # Last-resort default
        self.metrics[source] += 1
        self.metrics[f"category_{category}"] += 1

        return {
            "ticket": ticket,
            "category": category,
            "source": source,
            "latency_ms": round(latency_ms, 1),
        }

    def batch_classify(self, tickets: list[str]) -> list[dict]:
        return [self.classify(t) for t in tickets]

    def print_metrics(self):
        print("\n=== Classifier Metrics ===")
        total = self.metrics["slm"] + self.metrics["fallback_llm"]
        print(f"Total classified: {total}")
        for cat in self.CATEGORIES:
            count = self.metrics[f"category_{cat}"]
            print(f"  {cat:20s}: {count:3d} ({count/total:.0%} if total else 0)")
        if self.metrics["fallback_invoked"]:
            print(f"Fallback rate: {self.metrics['fallback_invoked']/total:.1%}")


# Run the classifier
classifier = TicketClassificationSystem(llm_client=llm)

test_tickets = [
    "API returns 404 on the /users endpoint since yesterday's deploy.",
    "Invoice #8821 shows $299 but I was promised $199 in my plan.",
    "Would love to see multi-language support in the next release.",
    "Do you have GDPR-compliant data processing agreements available?",
    "I've been waiting 2 weeks and still no resolution to my issue!",
    "Hi",  # Will be rejected by pre-processing
]

results = classifier.batch_classify(test_tickets)

print(f"{'Ticket':60s} | {'Category':20s} | {'Source':12s} | Latency")
print("-" * 115)
for r in results:
    ticket_preview = r['ticket'][:57] + '...' if len(r['ticket']) > 57 else r['ticket']
    category = r.get('category') or 'REJECTED'
    source = r.get('source', '')
    latency = f"{r.get('latency_ms', 0):.0f}ms" if r.get('latency_ms') else 'N/A'
    print(f"{ticket_preview:60s} | {category:20s} | {source:12s} | {latency}")

classifier.print_metrics()

## 7. Fine-Tuning with LoRA

**Approach:** Parameter-efficient fine-tuning with LoRA/QLoRA.

Details: See [Fine-Tuning Granite section](../../building_slm.md#working-with-granite-4-slm-family) in the tutorial

In [ ]:
# Fine-tuning setup — requires GPU with 8GB+ VRAM
# Install: pip install unsloth transformers datasets peft trl

FINETUNING_TEMPLATE = '''
# ── Fine-Tuning Granite 4.0 H-Micro with QLoRA ───────────────────────────

from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Step 1: Load model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="ibm-granite/granite-4.0-h-micro",
    max_seq_length=2048,
    load_in_4bit=True,      # QLoRA: 4-bit weights, fp16 activations
)

# Step 2: Add LoRA adapters (only ~1% of params become trainable)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # LoRA rank: 8–64 typical
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
)

# Step 3: Prepare your domain-specific dataset
# Format: instruction → output pairs
def format_example(row):
    return {
        "text": (
            f"### Instruction\nClassify this support ticket into one of: "
            f"Technical Issue, Billing Question, Feature Request, General Inquiry, Complaint.\n\n"
            f"### Input\n{row['ticket']}\n\n"
            f"### Response\n{row['category']}"
        )
    }

# Load your labeled data (minimum 100-500 examples recommended)
raw_data = [
    {"ticket": "Can\'t log in, error 500",    "category": "Technical Issue"},
    {"ticket": "Double charge on my card",     "category": "Billing Question"},
    # ... add your domain examples ...
]
dataset = Dataset.from_list([format_example(r) for r in raw_data])

# Step 4: Train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        output_dir="./granite_finetuned",
        logging_steps=10,
    ),
)
trainer.train()

# Step 5: Save and serve
model.save_pretrained("./granite_ticket_classifier")
tokenizer.save_pretrained("./granite_ticket_classifier")

# Optionally: push to HuggingFace Hub
# model.push_to_hub("your-org/granite-ticket-classifier")
'''

print("Fine-tuning template (run on GPU machine):")
print(FINETUNING_TEMPLATE)

## Summary

You've learned the core techniques for working with SLMs:

✅ Few-shot prompting for pattern matching <br> ✅ RAG for knowledge augmentation <br>
✅ Tool calling with minimal tool sets <br>
✅ Practical classification example <br>
✅ Fine-tuning with LoRA<br>

For comprehensive details: See [`building_slm.md`](../../building_slm.md)

**Next Steps:**
- Try with your own use case
- Experiment with different Granite models
- Fine-tune on domain-specific data
- Build subagent architectures